# Installation/Setup
- Imports the pipeline and plotting helpers from the local BE3D checkout
- Assumes DSSP, Clustal Omega, and MUSCLE are already installed locally (see the repo README)
- Pins Plotly's renderer to a single explicit value, so figures don't render once per auto-detected frontend


In [1]:
import os
import sys
import subprocess
import copy
import yaml
import pandas as pd
import numpy as np
import plotly.io as pio
from IPython.display import display, Image, SVG
from ipywidgets import Dropdown

# Force a SINGLE Plotly renderer -- when multiple Jupyter/Plotly frontend extensions are
# active (common in VS Code), Plotly's auto-detection can set pio.renderers.default to a
# combined string like "vscode+notebook_connected", so every fig.show() call renders once
# per registered renderer, stacking visible duplicates of every single plot in the
# notebook. Pinning to one explicit renderer avoids that regardless of what got detected.
pio.renderers.default = 'vscode'

BECLUST3D_PATH = '/Users/ymyung/Projects/BEClust3D/src/beclust3d-public'
sys.path.insert(0, BECLUST3D_PATH)
sys.path.insert(0, os.path.join(BECLUST3D_PATH, 'examples'))

# Reload the helper modules before importing from them: Python caches modules in
# sys.modules, so in a long-lived kernel a plain `from be3d_plotly import ...` keeps using
# the copy imported earlier in the session, and edits to the helper files (or newly added
# functions) are invisible until the kernel is restarted. Reloading here makes re-running
# this cell enough to pick them up.
import importlib
import be3d_local_helper
import be3d_plotly
importlib.reload(be3d_local_helper)
importlib.reload(be3d_plotly)

from be3d_local_helper import (
    show_svgs, show_images, plot_residue_dot, plot_ppi_vs_noppi_scatter,
    build_ppi_vs_noppi_scatter, show_molstar_viewer,
    chain_values_from_df, edit_yaml_widgets,
)
from be3d_plotly import (
    show_side_by_side, combine_side_by_side, show_stacked, show_figure_dropdown, plot_hypothesis_qa, plot_violin_by_processed_muttype, plot_score_scatter,
    plot_dendrogram, plot_meta_dendrogram, plot_lfc_lfc3d_scatter, plot_plddt_rsa_scatter,
    plot_domain_barplot, plot_plddt_dis_barplot, plot_enrichment_test,
    plot_meta_score_scatter, plot_meta_lfc_lfc3d_scatter, plot_meta_plddt_rsa_scatter,
    plot_meta_domain_barplot, plot_meta_plddt_dis_barplot,
    COLOR_POS, COLOR_NEG,
)

# Assumes DSSP, ClustalO, and MUSCLE are already installed locally (see the public repo's
# README for install instructions) -- unlike the Colab notebook, this one never shells out
# to apt-get/wget for setup.

def run_be3d(yaml_path):
    script = os.path.join(BECLUST3D_PATH, 'examples', 'be3d_local.py')
    # Capture + print explicitly rather than letting the child inherit stdout/stderr --
    # a subprocess's inherited file descriptors don't reliably show up in a notebook
    # cell's own output (Jupyter/Colab capture sys.stdout at the Python level, which a
    # child process's raw fd can bypass), so check=True alone can raise CalledProcessError
    # with no visible clue about what actually went wrong inside be3d_local.py.
    result = subprocess.run([sys.executable, script, yaml_path], capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
        result.check_returncode()

def load_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)

def run_be3d_if_needed(yaml_path, done_marker):
    if os.path.exists(done_marker):
        print(f'[skip] {done_marker} already exists')
    else:
        run_be3d(yaml_path)


# Settings
- Choose which mode to run: monomer, ppi, or blind_target
- Loads that mode's default YAML config
- Example gene: KBTBD4-HDAC1 (PDB 8VOJ), which has data to support all three modes


In [2]:
# KBTBD4-HDAC1 (8VOJ) supports all three modes: monomer, ppi (via ppi_diff), and blind_target.
MONOMER_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/KBTBD4_chain_B.yaml'
PPI_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/ppi_diff_KBTBD4_HDAC1.yaml'
BLIND_TARGET_YAML = '/Users/ymyung/Projects/BEClust3D/be3d_test/blind_target_KBTBD4_HDAC1.yaml'

for label, path in [('monomer', MONOMER_YAML), ('ppi (ppi_diff)', PPI_YAML), ('blind_target', BLIND_TARGET_YAML)]:
    cfg = load_yaml(path)
    print(f"--- {label}: mode='{cfg.get('mode')}' ---")
    shown = {k: cfg[k] for k in ('input_gene', 'input_uniprot', 'input_chain', 'output_dir') if k in cfg}
    print(yaml.safe_dump(shown, sort_keys=False))


--- monomer: mode='monomer' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/KBTBD4_chain_B

--- ppi (ppi_diff): mode='ppi_diff' ---
input_gene: KBTBD4, HDAC1
input_uniprot: Q9NVX7-2, Q13547
input_chain: B, C
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1

--- blind_target: mode='blind_target' ---
input_gene: KBTBD4
input_uniprot: Q9NVX7-2
input_chain: B
output_dir: /Users/ymyung/Projects/BEClust3D/be3d_test/output/blind_target/KBTBD4_HDAC1



## Select a mode
- Pick one of monomer / ppi / blind_target below
- Only the section(s) matching the selected mode actually run further down; the others print a skip message

In [11]:
import ipywidgets as widgets

MODE_OPTIONS = [
    ('Monomer -- single target gene, no PPI partner', 'monomer'),
    ('PPI -- target gene(s) compared with vs. without a PPI partner (mode: ppi_diff)', 'ppi'),
    ('Blind target -- target has no screen data of its own, scored purely from PPI partner(s)', 'blind_target'),
]

mode_selector = widgets.RadioButtons(
    options=MODE_OPTIONS, description='Mode:',
    style={'description_width': '60px'}, layout=widgets.Layout(width='750px'),
)

MODE = mode_selector.value

def _on_mode_change(change):
    global MODE
    MODE = change['new']
    print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")

mode_selector.observe(_on_mode_change, names='value')
display(mode_selector)
print(f"Selected mode: '{MODE}' -- re-run the config editor and the cells below for this mode.")


RadioButtons(description='Mode:', layout=Layout(width='750px'), options=(('Monomer -- single target gene, no P…

Selected mode: 'monomer' -- re-run the config editor and the cells below for this mode.


## Edit config for the selected mode
- Widgets for the selected mode's YAML fields, each with its own description
- Edits are written back to the YAML file as soon as you change a field, before the pipeline runs


In [18]:
YAML_BY_MODE = {'monomer': MONOMER_YAML, 'ppi': PPI_YAML, 'blind_target': BLIND_TARGET_YAML}

# Every field in the mode's yaml is shown below, in three groups:
#  - top (no header): the fields you'll change most often for a new gene/screen
#  - "Mutation categorization": how raw mut_col values collapse into categories, and the
#    priority order used when a guide hits more than one category
#  - "Advanced / other settings": everything else in the yaml (auto-filled -- nothing is
#    left unexposed), for the occasional case you need to tweak e.g. a p-value threshold
#    or the conservation-run settings
COMMON_IMPORTANT = ['input_gene', 'input_uniprot', 'input_chain', 'screen_dir', 'screens',
                     'output_dir', 'user_pdb', 'user_fasta', 'user_dssp']
MUT_CATEGORIZATION = ['database.mut_col', 'database.val_col', 'database.gene_col',
                       'database.edits_col', 'mutation_category', 'mutation_priority']

_blind_partners = load_yaml(BLIND_TARGET_YAML).get('partners', [])
BLIND_MUT_CATEGORIZATION = [
    f'partners[{i}].{field}'
    for i in range(len(_blind_partners))
    for field in ('mut_col', 'val_col', 'gene_col', 'edits_col', 'mut_categories', 'mutation_priority')
]

GROUPS_BY_MODE = {
    'monomer': [
        (None, COMMON_IMPORTANT),
        ('Mutation categorization', MUT_CATEGORIZATION),
        ('Advanced / other settings', None),
    ],
    'ppi': [
        (None, COMMON_IMPORTANT + ['score_type', 'skip_existing']),
        ('Mutation categorization', MUT_CATEGORIZATION),
        ('Advanced / other settings', None),
    ],
    'blind_target': [
        (None, ['input_gene', 'input_uniprot', 'input_chain', 'output_dir',
                'user_pdb', 'user_fasta', 'user_dssp']),
        ('Mutation categorization', BLIND_MUT_CATEGORIZATION),
        ('Advanced / other settings', None),
    ],
}

print(f"Editing config for mode '{MODE}' ({YAML_BY_MODE[MODE]}) -- "
      "changes below are written back to the yaml file immediately, picked up the next "
      "time a cell further down runs the pipeline. Every field in the yaml is shown: the "
      "most commonly changed ones at the top, mutation categorization in the middle, and "
      "everything else under 'Advanced / other settings'.")
edit_yaml_widgets(YAML_BY_MODE[MODE], GROUPS_BY_MODE[MODE])


Editing config for mode 'monomer' (/Users/ymyung/Projects/BEClust3D/be3d_test/KBTBD4_chain_B.yaml) -- changes below are written back to the yaml file immediately, picked up the next time a cell further down runs the pipeline. Nested settings (pthr, database, mutation_category, qa, ...) aren't exposed here -- edit the yaml file directly for those.


# BE-QA
- Runs the pipeline for the selected mode (skipped if its output already exists), then shows that mode's QA plots
- Kolmogorov-Smirnov and Mann-Whitney scatters already cover every screen in one figure, so they are shown once with no selection
- The processed per-guide LFC violin is the only per-screen plot, so it gets the single dropdown in this section
- ppi mode shows every gene at once (no-PPI leg, then PPI-mode leg); blind_target shows every partner at once
- The dropdown is on the plot itself and switches client-side -- no cell re-run


In [20]:
if MODE == 'monomer':
    monomer_cfg = load_yaml(YAML_BY_MODE['monomer'])
    monomer_dir, monomer_gene, monomer_uniprot = monomer_cfg['output_dir'], monomer_cfg['input_gene'], monomer_cfg['input_uniprot']
    run_be3d_if_needed(YAML_BY_MODE['monomer'], os.path.join(monomer_dir, 'RUN_COMPLETED.txt'))

    monomer_screens = [s.strip().split('.')[0] for s in monomer_cfg['screens'].split(',')]

    # The KS/MW hypothesis-test plots already cover every screen in one figure, so they need
    # no selection at all and are just shown once.
    print('QA -- Kolmogorov-Smirnov and Mann-Whitney (all screens):')
    show_side_by_side(
        plot_hypothesis_qa(monomer_dir, test='KolmogorovSmirnov'),
        plot_hypothesis_qa(monomer_dir, test='MannWhitney'),
        width=600, height=400, spacing=0.08
    )

    # Only the violin is per-screen, so it gets the one dropdown in this section. Every screen's
    # violin is built up front and switched by Plotly's own in-figure dropdown
    # (layout.updatemenus), so changing screen does NOT re-run this cell.
    print('Processed LFC distribution by mutation category (violin, post mutation_priority + '
          'per-category filtering) -- pick a screen on the plot:')
    show_figure_dropdown(
        {s: plot_violin_by_processed_muttype(monomer_dir, monomer_gene, s, width=600, height=400)
         for s in monomer_screens},
        description='Screen:'
    )

elif MODE == 'ppi':
    ppi_cfg = load_yaml(YAML_BY_MODE['ppi'])
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # The QA/violin plots need the per-gene no_ppi and ppi legs to already exist, so run the
    # pipeline first (mirrors the BE-Clust3D (PPI) cell further down). Variant yamls go to the
    # scratch dir, not the cloned repo's yaml dir (see Settings cell comment). skip_existing
    # makes re-runs a no-op.
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join(os.path.dirname(YAML_BY_MODE['ppi']), f'_ppi_diff_{score_type}.yaml')
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')

    # No gene dropdown: the hypothesis-test plots cover all screens already, and every gene is
    # shown at once (no-PPI leg then PPI-mode leg per gene).
    for _gene in gene_names:
        _noppi, _ppi = os.path.join(ppi_root, 'no_ppi', _gene), os.path.join(ppi_root, 'ppi', _gene)
        print(f'{_gene} -- QA (KS2 test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(_noppi, test='KolmogorovSmirnov'),
            plot_hypothesis_qa(_ppi, test='KolmogorovSmirnov'),
            width=600, height=400,
        )
        print(f'{_gene} -- QA (MW test), no-PPI then PPI-mode:')
        show_side_by_side(
            plot_hypothesis_qa(_noppi, test='MannWhitney'),
            plot_hypothesis_qa(_ppi, test='MannWhitney'),
            width=600, height=400,
        )

    # One dropdown, for the violin only -- each variant is a row of (no-PPI, PPI-mode) panels
    # per gene for that screen. Switching is client-side; no cell re-run.
    print('Processed LFC distribution by mutation category (violin, post mutation_priority + '
          'per-category filtering) -- no-PPI then PPI-mode for each gene; pick a screen on the plot:')
    show_figure_dropdown(
        {
            s: combine_side_by_side(*[
                f for _gene in gene_names for f in (
                    plot_violin_by_processed_muttype(os.path.join(ppi_root, 'no_ppi', _gene), _gene, s),
                    plot_violin_by_processed_muttype(os.path.join(ppi_root, 'ppi', _gene), _gene, s),
                )
            ], width=450, height=400)
            for s in ppi_screens
        },
        description='Screen:'
    )

else:
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_partners = blind_cfg['partners']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    # blind_target's target itself never gets hypothesis_test/screendata (run_blind_target skips
    # both -- it has no screen data of its own). Each partner runs through parse_be_data +
    # prioritize_by_sequence (preprocess_ppi_partner), same as a monomer run, but
    # preprocess_ppi_partner explicitly skips hypothesis_test -- so there's no KS2/MW QA to show
    # even for the partner, just its processed-LFC violin.
    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    partner_by_gene = {p['gene']: p for p in blind_partners}
    partner_screens_by_gene = {
        g: [s.strip().split('.')[0] for s in p['screens'].split(',')]
        for g, p in partner_by_gene.items()
    }
    all_partner_screens = sorted({s for screens in partner_screens_by_gene.values() for s in screens})

    print('Note: blind_target partners skip hypothesis_test (preprocess_ppi_partner), so no '
          "KS2/MW QA plot is available -- showing each partner's processed LFC distribution "
          '(violin) instead. No partner dropdown: every partner is shown side by side. '
          'Pick a screen on the plot:')

    def _partner_violin(partner_gene, screen):
        if screen not in partner_screens_by_gene[partner_gene]:
            return None
        chain = partner_by_gene[partner_gene]['chain']
        pdir = os.path.join(blind_dir, 'ppi_partners', f'{partner_gene}_chain_{chain}')
        return plot_violin_by_processed_muttype(pdir, partner_gene, screen)

    show_figure_dropdown(
        {
            s: combine_side_by_side(*[_partner_violin(g, s) for g in partner_by_gene],
                                    width=600, height=400)
            for s in all_partner_screens
        },
        description='Screen:'
    )


[skip] /Users/ymyung/Projects/BEClust3D/be3d_test/output/KBTBD4_chain_B/RUN_COMPLETED.txt already exists
QA -- Kolmogorov-Smirnov and Mann-Whitney (all screens):


Processed LFC distribution by mutation category (violin, post mutation_priority + per-category filtering) -- pick a screen on the plot:


# Monomer Mode


## BE-Clust3D
- Residue dot-plots of LFC and LFC3D (positive and negative shown separately)
- LFC vs. LFC3D scatter, highlighting residues that have an LFC3D value but no direct LFC value
- pLDDT vs. RSA scatter, LFC3D hit count by protein domain, and hit count by pLDDT-disorder category
- Enrichment test (log2 odds ratio) for pLDDT-disorder category
- Each plot group carries its own screen dropdown, on the plot itself -- switching screens does not re-run the cell


In [21]:
if MODE == 'monomer':
    # Every dropdown in this cell is Plotly's own in-figure dropdown (layout.updatemenus): all
    # screens are built up front and switching between them is client-side trace toggling, so
    # picking a different screen does NOT re-run the cell. One dropdown per plot group, since
    # each group is its own figure.
    _S = monomer_screens

    print('BE-Clust3D -- residue dot-plots, LFC (positive, negative):')
    show_figure_dropdown({s: combine_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, s, score_type='LFC', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, s, score_type='LFC', direction='negative'),
        width=600, height=400) for s in _S}, description='BE-Clust3D screen:')

    print('BE-Clust3D -- residue dot-plots, LFC3D (positive, negative):')
    show_figure_dropdown({s: combine_side_by_side(
        plot_score_scatter(monomer_dir, monomer_gene, s, score_type='LFC3D', direction='positive'),
        plot_score_scatter(monomer_dir, monomer_gene, s, score_type='LFC3D', direction='negative'),
        width=600, height=400) for s in _S}, description='BE-Clust3D screen:')

    print('BE-Clust3D -- LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):')
    show_figure_dropdown({s: plot_lfc_lfc3d_scatter(monomer_dir, monomer_gene, s, width=500, height=400)
                          for s in _S}, description='BE-Clust3D screen:')

    print('BE-Clust3D -- pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category:')
    show_figure_dropdown({s: combine_side_by_side(
        plot_plddt_rsa_scatter(monomer_dir, monomer_gene, s),
        plot_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, s),
        plot_plddt_dis_barplot(monomer_dir, monomer_gene, s),
        width=600, height=400) for s in _S}, description='BE-Clust3D screen:')

    print('BE-Clust3D -- enrichment test (pLDDT-disorder, log2 odds ratio):')
    show_figure_dropdown({s: plot_enrichment_test(monomer_dir, monomer_gene, screen_name=s, width=600, height=300)
                          for s in _S}, description='BE-Clust3D screen:')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D (monomer).")


BE-Clust3D -- residue dot-plots, LFC (positive, negative):


BE-Clust3D -- residue dot-plots, LFC3D (positive, negative):


BE-Clust3D -- LFC vs. LFC3D (residues with LFC3D but no LFC shown in the left strip):


BE-Clust3D -- pLDDT vs. RSA / LFC3D hit count by domain / pLDDT-disorder category:
[be3d_plotly] No domain annotations available for KBTBD4 ('Domain' is empty/NaN for every residue -- e.g. UniProt has no queried domain features for this entry) — skipping domain barplot.
[be3d_plotly] No domain annotations available for KBTBD4 ('Domain' is empty/NaN for every residue -- e.g. UniProt has no queried domain features for this entry) — skipping domain barplot.


BE-Clust3D -- enrichment test (pLDDT-disorder, log2 odds ratio):


### Dendrogram
- Spatial hierarchical clustering of the significant residues (p<0.05), rendered as a merge tree
- Screen, score type and direction are all in one dropdown on the plot; switching does not re-run the cell
- Combinations with no significant residues are left out of the dropdown and reported in a note


In [ ]:
if MODE == 'monomer':
    # Screen and score-type/direction are both in one in-figure dropdown, so any switch is
    # client-side and needs no cell re-run.
    print('BE-Clust3D -- dendrograms (p<0.05); pick screen / score type / direction on the plot:')
    show_figure_dropdown({
        f'{s} | {label}': plot_dendrogram(monomer_dir, monomer_gene, s, score_type=st,
                                          direction=d, height=400)
        for s in monomer_screens
        for label, (st, d) in {
            'LFC positive': ('LFC', 'Positive'), 'LFC negative': ('LFC', 'Negative'),
            'LFC3D positive': ('LFC3D', 'Positive'), 'LFC3D negative': ('LFC3D', 'Negative'),
        }.items()
    }, description='BE-Clust3D dendrogram:')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-Clust3D dendrogram (monomer).")


## BE-MetaClust3D
- Only shown when the gene has multiple screens (otherwise there is nothing to meta-aggregate)
- Same plots as BE-Clust3D above, but computed on the meta-aggregated score across all screens instead of one screen at a time


In [ ]:
if MODE == 'monomer':
    monomer_func_meta = monomer_cfg['function_for_meta']

    if len(monomer_screens) > 1:
        print('Meta residue dot-plots, meta-LFC (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC', direction='negative'),
            width=600, height=400
        )
        print('Meta residue dot-plots, meta-LFC3D (positive, negative):')
        show_side_by_side(
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='positive'),
            plot_meta_score_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, score_type='LFC3D', direction='negative'),
            width=600, height=400
        )

        print('meta-LFC vs. meta-LFC3D (residues with meta-LFC3D but no meta-LFC shown in the left strip):')
        fig = plot_meta_lfc_lfc3d_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta, width=500, height=400)
        if fig is not None:
            display(fig)

        print('pLDDT vs. RSA / Meta LFC3D hit count by domain / pLDDT-disorder category:')
        show_side_by_side(
            plot_meta_plddt_rsa_scatter(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            plot_meta_domain_barplot(monomer_dir, monomer_gene, monomer_uniprot, function_for_meta=monomer_func_meta),
            plot_meta_plddt_dis_barplot(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta),
            width=600, height=400
        )

        print('Enrichment test (pLDDT-disorder, log2 odds ratio):')
        fig = plot_enrichment_test(monomer_dir, monomer_gene, screen_name=None, width=400, height=300)
        if fig is not None:
            display(fig)
    else:
        print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D (monomer).")


### Meta dendrogram
- Meta-aggregated equivalent of the dendrogram above (no screen selection -- meta scores aggregate every screen)
- Switch score type / direction with the dropdown on the plot; no cell re-run


In [ ]:
if MODE == 'monomer' and len(monomer_screens) > 1:
    # Score type / direction in one in-figure dropdown -- client-side, no re-run. Meta scores
    # aggregate every screen, so there is no screen selection here.
    print('BE-MetaClust3D -- meta dendrograms (p<0.05); pick score type / direction on the plot:')
    show_figure_dropdown({
        label: plot_meta_dendrogram(monomer_dir, monomer_gene, function_for_meta=monomer_func_meta,
                                    score_type=st, direction=d, height=400)
        for label, (st, d) in {
            'Meta LFC positive': ('LFC', 'Positive'), 'Meta LFC negative': ('LFC', 'Negative'),
            'Meta LFC3D positive': ('LFC3D', 'Positive'), 'Meta LFC3D negative': ('LFC3D', 'Negative'),
        }.items()
    }, description='BE-MetaClust3D dendrogram:')
elif MODE == 'monomer':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'monomer' -- skipping BE-MetaClust3D dendrogram (monomer).")


# PPI mode


## BE-Clust3D
- Runs the ppi_diff pipeline (PPI leg + no-PPI leg, then merge); re-running is cheap because completed steps are skipped
- Residue dot-plots of LFC3D (positive and negative), no-PPI leg then PPI-mode leg side by side
- LFC vs. LFC3D, pLDDT vs. RSA, hit count by pLDDT-disorder category, and the enrichment test, each no-PPI then PPI-mode
- no-PPI LFC3D (x) vs. PPI-mode LFC3D (y) scatter for every gene/screen
- Each plot group carries its own gene | screen dropdown, on the plot itself -- switching does not re-run the cell


In [4]:
if MODE == 'ppi':
    ppi_cfg = load_yaml(PPI_YAML)
    ppi_root = ppi_cfg['output_dir']
    gene_names = [g.strip() for g in ppi_cfg['input_gene'].split(',')]
    chain_list = [c.strip() for c in ppi_cfg['input_chain'].split(',')]
    ppi_screens = [s.strip().split('.')[0] for s in ppi_cfg['screens'].split(',')]

    # mode: ppi_diff runs the PPI leg (mode: complex) and the no-PPI leg (mode: monomer, per
    # gene) once, then merges -- skip_existing makes each pass a no-op once the first has run.
    # score_type controls only the (cheap) merge step: 'LFC3D' produces one merged TSV+PDB set
    # per screen, 'Meta_LFC3D' the meta-aggregated one (needed by BE-MetaClust3D and
    # Merged results below).
    def run_ppi_diff_pass(score_type):
        variant_cfg = copy.deepcopy(ppi_cfg)
        variant_cfg['score_type'] = score_type
        variant_yaml = os.path.join(os.path.dirname(PPI_YAML), f'_ppi_diff_{score_type}.yaml')
        with open(variant_yaml, 'w') as f:
            yaml.safe_dump(variant_cfg, f)
        run_be3d(variant_yaml)

    run_ppi_diff_pass('LFC3D')
    run_ppi_diff_pass('Meta_LFC3D')

    ppi_func_meta = ppi_cfg['function_for_meta']
    ppi_uniprot_by_gene = dict(zip(gene_names, [u.strip() for u in ppi_cfg['input_uniprot'].split(',')]))
    _chain_by_gene = dict(zip(gene_names, chain_list))

    # Gene and screen are combined into one in-figure dropdown per plot group -- every
    # combination is built up front, so switching is client-side and needs no cell re-run.
    _GS = [(g, s) for g in gene_names for s in ppi_screens]

    def _key(g, s):
        return f'{g} (chain {_chain_by_gene[g]}) | {s}'

    def _dirs(g):
        return os.path.join(ppi_root, 'no_ppi', g), os.path.join(ppi_root, 'ppi', g)

    print('BE-Clust3D -- residue dot-plots, LFC3D positive -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_score_scatter(_dirs(g)[0], g, s, score_type='LFC3D', direction='positive'),
        plot_score_scatter(_dirs(g)[1], g, s, score_type='LFC3D', direction='positive'),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_score_scatter(_dirs(g)[0], g, s, score_type='LFC3D', direction='negative'),
        plot_score_scatter(_dirs(g)[1], g, s, score_type='LFC3D', direction='negative'),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- LFC vs. LFC3D -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_lfc_lfc3d_scatter(_dirs(g)[0], g, s),
        plot_lfc_lfc3d_scatter(_dirs(g)[1], g, s),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):')
    _ppi_vs_noppi = {s: pd.read_csv(os.path.join(ppi_root, f'ppi_vs_noppi_{s}.tsv'), sep='\t')
                     for s in ppi_screens}
    show_figure_dropdown({_key(g, s): build_ppi_vs_noppi_scatter(
        _ppi_vs_noppi[s][_ppi_vs_noppi[s]['gene'] == g], 'LFC3D', width=500, height=400)
        for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- pLDDT vs. RSA -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_plddt_rsa_scatter(_dirs(g)[0], g, s),
        plot_plddt_rsa_scatter(_dirs(g)[1], g, s),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_plddt_dis_barplot(_dirs(g)[0], g, s),
        plot_plddt_dis_barplot(_dirs(g)[1], g, s),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')

    print('BE-Clust3D -- enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g, s): combine_side_by_side(
        plot_enrichment_test(_dirs(g)[0], g, screen_name=s),
        plot_enrichment_test(_dirs(g)[1], g, screen_name=s),
        width=600, height=400) for g, s in _GS}, description='BE-Clust3D gene | screen:')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D (ppi).")


[ppi_diff] skipping PPI leg, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi
[ppi_diff] skipping no-PPI leg for KBTBD4, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/KBTBD4
[ppi_diff] skipping no-PPI leg for HDAC1, already completed in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/no_ppi/HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_abe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_abe_neg_control.pdb, ppi_abe_neg_control.pdb, delta_abe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1
wrote /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/ppi_vs_noppi_cbe_neg_control.tsv (1016 residues across 2 chains)
wrote noppi_cbe_neg_control.pdb, ppi_cbe_neg_control.pdb, delta_cbe_neg_control.pdb in /Users/ymyung/Projects/BEClust3D/be3d_test/o

BE-Clust3D -- residue dot-plots, LFC3D negative -- no-PPI, then PPI-mode:


BE-Clust3D -- LFC vs. LFC3D -- no-PPI, then PPI-mode:


BE-Clust3D -- no-PPI LFC3D (x) vs. PPI-mode LFC3D (y):


BE-Clust3D -- pLDDT vs. RSA -- no-PPI, then PPI-mode:


BE-Clust3D -- LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:


BE-Clust3D -- enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:


### Dendrogram
- LFC3D spatial clustering merge tree
- Gene, screen, leg (no-PPI / PPI-mode) and direction are all in one dropdown on the plot; no cell re-run


In [5]:
if MODE == 'ppi':
    # Gene, screen, leg and direction all in one in-figure dropdown -- client-side, no re-run.
    print('BE-Clust3D -- LFC3D dendrograms (p<0.05); pick gene / screen / leg / direction on the plot:')
    show_figure_dropdown({
        f'{g} | {s} | {label}': plot_dendrogram(
            (os.path.join(ppi_root, 'no_ppi', g) if leg == 'no-PPI' else os.path.join(ppi_root, 'ppi', g)),
            g, s, score_type='LFC3D', direction=d, height=400)
        for g in gene_names
        for s in ppi_screens
        for label, (leg, d) in {
            'no-PPI positive': ('no-PPI', 'Positive'), 'PPI-mode positive': ('ppi', 'Positive'),
            'no-PPI negative': ('no-PPI', 'Negative'), 'PPI-mode negative': ('ppi', 'Negative'),
        }.items()
    }, description='BE-Clust3D dendrogram:')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-Clust3D dendrogram (ppi).")


BE-Clust3D -- LFC3D dendrograms (p<0.05); pick gene / screen / leg / direction on the plot:


## BE-MetaClust3D
- Only shown when the gene has multiple screens
- Same no-PPI vs. PPI-mode comparisons as BE-Clust3D (ppi) above, but on the meta-aggregated meta-LFC3D score
- Each plot group carries its own gene dropdown, on the plot itself -- no screen dropdown, since meta scores already aggregate every screen


In [6]:
if MODE == 'ppi' and len(ppi_screens) > 1:
    df_meta = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')

    # One in-figure gene dropdown per plot group -- client-side, no cell re-run. Meta scores
    # already aggregate every screen, so there is no screen selection here.
    def _dirs(g):
        return os.path.join(ppi_root, 'no_ppi', g), os.path.join(ppi_root, 'ppi', g)

    def _key(g):
        return f'{g} (chain {_chain_by_gene[g]})'

    print('BE-MetaClust3D -- meta residue dot-plots, meta-LFC3D positive -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_meta_score_scatter(_dirs(g)[0], g, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
        plot_meta_score_scatter(_dirs(g)[1], g, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='positive'),
        width=600, height=400) for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- meta residue dot-plots, meta-LFC3D negative -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_meta_score_scatter(_dirs(g)[0], g, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
        plot_meta_score_scatter(_dirs(g)[1], g, function_for_meta=ppi_func_meta, score_type='LFC3D', direction='negative'),
        width=600, height=400) for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- meta-LFC vs. meta-LFC3D -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_meta_lfc_lfc3d_scatter(_dirs(g)[0], g, function_for_meta=ppi_func_meta),
        plot_meta_lfc_lfc3d_scatter(_dirs(g)[1], g, function_for_meta=ppi_func_meta),
        width=500, height=500) for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- no-PPI meta-LFC3D (x) vs. PPI-mode meta-LFC3D (y):')
    show_figure_dropdown({_key(g): build_ppi_vs_noppi_scatter(
        df_meta[df_meta['gene'] == g], 'meta-LFC3D', width=500, height=400)
        for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- pLDDT vs. RSA -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_meta_plddt_rsa_scatter(_dirs(g)[0], g, function_for_meta=ppi_func_meta),
        plot_meta_plddt_rsa_scatter(_dirs(g)[1], g, function_for_meta=ppi_func_meta),
        width=600, height=400) for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- meta LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_meta_plddt_dis_barplot(_dirs(g)[0], g, function_for_meta=ppi_func_meta),
        plot_meta_plddt_dis_barplot(_dirs(g)[1], g, function_for_meta=ppi_func_meta),
        width=600, height=400) for g in gene_names}, description='BE-MetaClust3D gene:')

    print('BE-MetaClust3D -- meta enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:')
    show_figure_dropdown({_key(g): combine_side_by_side(
        plot_enrichment_test(_dirs(g)[0], g, screen_name=None),
        plot_enrichment_test(_dirs(g)[1], g, screen_name=None),
        width=600, height=400) for g in gene_names}, description='BE-MetaClust3D gene:')
elif MODE == 'ppi':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D (ppi).")


BE-MetaClust3D -- meta residue dot-plots, meta-LFC3D positive -- no-PPI, then PPI-mode:


BE-MetaClust3D -- meta residue dot-plots, meta-LFC3D negative -- no-PPI, then PPI-mode:


BE-MetaClust3D -- meta-LFC vs. meta-LFC3D -- no-PPI, then PPI-mode:


BE-MetaClust3D -- no-PPI meta-LFC3D (x) vs. PPI-mode meta-LFC3D (y):


BE-MetaClust3D -- pLDDT vs. RSA -- no-PPI, then PPI-mode:


BE-MetaClust3D -- meta LFC3D hit count by pLDDT-disorder category -- no-PPI, then PPI-mode:


BE-MetaClust3D -- meta enrichment test (pLDDT-disorder) -- no-PPI, then PPI-mode:


### Meta dendrogram
- Meta-aggregated equivalent of the dendrogram above
- Gene, leg and direction in one dropdown on the plot; no cell re-run


In [7]:
if MODE == 'ppi' and len(ppi_screens) > 1:
    # Gene, leg and direction in one in-figure dropdown -- client-side, no re-run.
    print('BE-MetaClust3D -- meta LFC3D dendrograms (p<0.05); pick gene / leg / direction on the plot:')
    show_figure_dropdown({
        f'{g} | {label}': plot_meta_dendrogram(
            (os.path.join(ppi_root, 'no_ppi', g) if leg == 'no-PPI' else os.path.join(ppi_root, 'ppi', g)),
            g, function_for_meta=ppi_func_meta, score_type='LFC3D', direction=d, height=400)
        for g in gene_names
        for label, (leg, d) in {
            'no-PPI positive': ('no-PPI', 'Positive'), 'PPI-mode positive': ('ppi', 'Positive'),
            'no-PPI negative': ('no-PPI', 'Negative'), 'PPI-mode negative': ('ppi', 'Negative'),
        }.items()
    }, description='BE-MetaClust3D dendrogram:')
elif MODE == 'ppi':
    print('Only one screen -- no meta-aggregation to show.')
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping BE-MetaClust3D dendrogram (ppi).")


BE-MetaClust3D -- meta LFC3D dendrograms (p<0.05); pick gene / leg / direction on the plot:


## Merged results (meta-LFC3D)
- Table of the top 10 residues ranked by |delta meta-LFC3D| (PPI-mode minus no-PPI)
- Molstar structure viewer colored by the selected view: no-PPI meta-LFC3D, PPI-mode meta-LFC3D, or their delta (-2 to +2, white at 0)
- The top 10 |delta| residues from the table are highlighted as spheres
- Table and viewer are separate cells -- Molstar's widget otherwise sits on top of and hides the table


In [8]:
if MODE == 'ppi':
    df_merged = pd.read_csv(os.path.join(ppi_root, 'ppi_vs_noppi_Meta_LFC3D.tsv'), sep='\t')
    df_merged_sorted = df_merged.reindex(df_merged['delta_score'].abs().sort_values(ascending=False).index)

    print('Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):')
    top10 = df_merged_sorted.head(10)
    display(top10[['gene', 'chain', 'unipos', 'unires', 'noppi_score', 'ppi_score', 'delta_score']])
    top10_unipos = top10['unipos'].tolist()
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results.")


Top 10 residues by |delta meta-LFC3D| (PPI - no-PPI):


,gene,chain,unipos,unires,noppi_score,ppi_score,delta_score
330,KBTBD4,B,331,P,-0.055402,2.265313,2.320715
331,KBTBD4,B,332,R,0.262081,2.140287,1.878206
332,KBTBD4,B,333,D,0.394308,2.272514,1.878206
31,KBTBD4,B,32,F,1.290539,-0.247093,-1.537632
33,KBTBD4,B,34,N,1.647224,1.030698,-0.616526
121,KBTBD4,B,122,G,0.466410,0.921791,0.455381
125,KBTBD4,B,126,L,-0.241375,0.169219,0.410594
511,KBTBD4,B,512,A,-0.583967,-0.199322,0.384645
26,KBTBD4,B,27,M,-0.414375,-0.047508,0.366868
25,KBTBD4,B,26,S,-0.414375,-0.047508,0.366868


In [9]:
if MODE == 'ppi':
    base_pdb = os.path.join(ppi_root, 'ppi', gene_names[0], 'sequence_structure')
    base_pdb = os.path.join(base_pdb, [f for f in os.listdir(base_pdb) if f.endswith('_processed.pdb')][0])

    merged_views = {
        'Delta (PPI - no-PPI) meta-LFC3D': chain_values_from_df(df_merged, 'delta_score'),
        'No-PPI meta-LFC3D': chain_values_from_df(df_merged, 'noppi_score'),
        'PPI-mode meta-LFC3D': chain_values_from_df(df_merged, 'ppi_score'),
    }

    # Viewer and its selector are one unit: the dropdown sits in the viewer's own toolbar, and
    # switching is handled in the browser (no kernel round-trip, no ipywidgets). That is also
    # why this works where the PDBeMolstar widget did not -- see show_molstar_viewer().
    show_molstar_viewer(
        base_pdb, merged_views, vmax=2.0, highlight_top_n=10, title='Color by:', height=460,
        caption='blue = positive, red = negative, grey = no value; spheres = top 10 |delta| residues',
        save_to=os.path.join(ppi_root, 'merged_meta_LFC3D_viewer.html'),
    )
else:
    print(f"[skipped] mode is '{MODE}', not 'ppi' -- skipping PPI merged results (structure view).")


[html] standalone viewer: /Users/ymyung/Projects/BEClust3D/be3d_test/output/ppi_diff_KBTBD4_HDAC1/merged_meta_LFC3D_viewer.html


/Users/ymyung/miniforge3/envs/beclust3d_arm/lib/python3.10/site-packages/IPython/core/display.py:475: UserWarning:

Consider using IPython.display.IFrame instead



# Blind target mode
- Table of residues that received a blind LFC3D value (meta-aggregated column when the target has multiple partner screens, otherwise the single screen's column)
- Residue dot-plot of that same signed blind LFC3D value
- Molstar 3D structure viewer colored by the selected view (Negative / Positive / Overall); blue marks positive values, red marks negative ones
- Table, dot-plot, and viewer are separate cells -- Molstar's widget otherwise sits on top of and hides the table


In [15]:
if MODE == 'blind_target':
    blind_cfg = load_yaml(YAML_BY_MODE['blind_target'])
    blind_dir = blind_cfg['output_dir']
    blind_gene, blind_chain = blind_cfg['input_gene'], blind_cfg['input_chain']
    blind_tsv = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D.tsv')

    if not os.path.exists(blind_tsv):
        run_be3d(YAML_BY_MODE['blind_target'])

    df_blind = pd.read_csv(blind_tsv, sep='\t')

    # use the meta-aggregated column if there's more than one partner screen, else the single screen's
    screen_names = [c[:-len('_LFC3D_blind_overall')] for c in df_blind.columns if c.endswith('_LFC3D_blind_overall')]
    if 'Meta_LFC3D_blind_overall' in df_blind.columns:
        neg_col, pos_col, overall_col = 'Meta_LFC3D_blind_neg', 'Meta_LFC3D_blind_pos', 'Meta_LFC3D_blind_overall'
    else:
        screen_name = screen_names[0]
        neg_col, pos_col, overall_col = f'{screen_name}_LFC3D_blind_neg', f'{screen_name}_LFC3D_blind_pos', f'{screen_name}_LFC3D_blind_overall'

    df_blind_hits = df_blind[(df_blind[neg_col] != '-') | (df_blind[pos_col] != '-')]
    print(f'{blind_gene} (chain {blind_chain}) -- {len(df_blind_hits)}/{len(df_blind)} residues have a blind LFC3D value:')
    display(df_blind_hits[['unipos', 'unires', 'chain', neg_col, pos_col, overall_col]])
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results.")


KBTBD4 (chain B) -- 4/534 residues have a blind LFC3D value:


,unipos,unires,chain,Meta_LFC3D_blind_neg,Meta_LFC3D_blind_pos,Meta_LFC3D_blind_overall
330,331,P,B,-,2.390828716620627,2.390828716620627
331,332,R,B,-,1.878206121495052,1.878206121495052
332,333,D,B,-,1.878206121495052,1.878206121495052
372,373,V,B,-,0.2389496730824325,0.2389496730824325


In [16]:
if MODE == 'blind_target':
    print('Residue dot-plot (signed value: negative or positive column, whichever is set):')
    signed = pd.to_numeric(df_blind[neg_col].replace('-', pd.NA), errors='coerce')
    signed = signed.fillna(pd.to_numeric(df_blind[pos_col].replace('-', pd.NA), errors='coerce'))
    df_blind_signed = df_blind.copy()
    df_blind_signed['_signed_blind_LFC3D'] = signed
    plot_residue_dot(df_blind_signed, '_signed_blind_LFC3D', f'{blind_gene} blind LFC3D')


Residue dot-plot (signed value: negative or positive column, whichever is set):


In [17]:
if MODE == 'blind_target':
    overall_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_overall.pdb')
    pos_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_pos.pdb')
    neg_pdb = os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_neg.pdb')
    # run_blind_target always writes all three PDBs together (same coordinates, different
    # B-factors baked in) when user_pdb is set, so which one is loaded as the base structure
    # doesn't matter -- only the coloring changes per selection.
    blind_base_pdb = overall_pdb if os.path.exists(overall_pdb) else (pos_pdb if os.path.exists(pos_pdb) else neg_pdb)

    def _blind_chain_values(col):
        values = pd.to_numeric(df_blind[col].replace('-', pd.NA), errors='coerce')
        return {blind_chain: {int(p): float(v) for p, v in zip(df_blind['unipos'], values) if pd.notna(v)}}

    blind_views = {
        'Overall': _blind_chain_values(overall_col),
        'Negative': _blind_chain_values(neg_col),
        'Positive': _blind_chain_values(pos_col),
    }

    # Selector lives in the viewer's toolbar -- see the merged-results structure cell.
    show_molstar_viewer(
        blind_base_pdb, blind_views, vmax=2.0, highlight_top_n=10, title='Color by:', height=460,
        caption='blue = positive, red = negative, grey = no value',
        save_to=os.path.join(blind_dir, f'{blind_gene}_{blind_chain}_blind_LFC3D_viewer.html'),
    )
else:
    print(f"[skipped] mode is '{MODE}', not 'blind_target' -- skipping blind-target results (structure view).")


[html] standalone viewer: /Users/ymyung/Projects/BEClust3D/be3d_test/output/blind_target/KBTBD4_HDAC1/KBTBD4_B_blind_LFC3D_viewer.html


/Users/ymyung/miniforge3/envs/beclust3d_arm/lib/python3.10/site-packages/IPython/core/display.py:475: UserWarning:

Consider using IPython.display.IFrame instead

